# Byte-Pair Encoding, from scratch

**The whole idea in one sentence:** start with the smallest possible pieces, then repeatedly glue together whichever adjacent pair shows up most often.

That's genuinely it. The list of glue operations you performed, *in order*, **is** the tokenizer.

By the end of this notebook you'll have:

- trained a BPE tokenizer by hand on a five-word corpus, watching every merge
- written a real trainer, encoder, and decoder in ~40 lines
- seen why merge **order** matters, with a case where the obvious alternative gives a different answer
- seen why decoding has exactly one trap, and hit it deliberately
- trained on real text and measured compression against GPT-2's tokenizer

Nothing here imports from `litterbox` — the point is to build it in the open. Once you've written the real thing in `src/litterbox/data/tokenizer/`, this notebook stays as the explanation.

In [ ]:
from collections import Counter

def render(seq, vocab):
    """Show a sequence of token ids as readable pieces, space-separated."""
    return " ".join(vocab[i].decode("utf-8", errors="replace") for i in seq)

---
## 1. Training, traced by hand

A corpus of five words. The counts are what matter — BPE never looks at word *order*, only at how often each pair of adjacent symbols appears.

In [ ]:
word_freqs = {
    "hug":  10,
    "pug":   5,
    "pun":  12,
    "bun":   4,
    "hugs":  5,
}
word_freqs

### Start: everything is bytes

Not characters — **bytes**. Ids `0-255` are handed to us for free, one per possible byte value. That's the starting vocabulary, and it means every possible input is already representable.

In [ ]:
vocab  = {i: bytes([i]) for i in range(256)}
splits = {w: tuple(w.encode("utf-8")) for w in word_freqs}

for w, seq in splits.items():
    print(f"{w:6} x{word_freqs[w]:<3}  {render(seq, vocab):12}  ids={list(seq)}")

Those ids are just ASCII: `h`=104, `u`=117, `g`=103, `s`=115, `p`=112, `n`=110, `b`=98.

### Count every adjacent pair

Weighted by how often the word appears — `hug` contributes its pairs 10 times, not once. This weighting is the single most important performance decision in the whole algorithm, and we'll come back to it.

In [ ]:
def count_pairs(splits, freqs):
    pairs = Counter()
    for word, seq in splits.items():
        for pair in zip(seq, seq[1:]):
            pairs[pair] += freqs[word]
    return pairs

pairs = count_pairs(splits, word_freqs)

for (a, b), n in pairs.most_common():
    print(f"{vocab[a].decode():>3} + {vocab[b].decode():<3}  ->  {n}")

`u g` wins with 20. Glue it into a brand-new token.

The new token gets id **256** — the first id above the byte range. Merge #1 creates token #256, merge #2 creates #257, and so on. This is the part worth internalising: **the merge list and the vocabulary are the same object seen from two directions.**

In [ ]:
def merge_seq(seq, pair, new_id):
    """Replace every occurrence of `pair` in `seq` with `new_id`."""
    out, i = [], 0
    while i < len(seq):
        if i < len(seq) - 1 and (seq[i], seq[i + 1]) == pair:
            out.append(new_id)
            i += 2
        else:
            out.append(seq[i])
            i += 1
    return tuple(out)


best   = pairs.most_common(1)[0][0]
new_id = 256

vocab[new_id] = vocab[best[0]] + vocab[best[1]]
splits = {w: merge_seq(s, best, new_id) for w, s in splits.items()}

print(f"merge 1:  {vocab[best[0]].decode()} + {vocab[best[1]].decode()} -> {vocab[new_id].decode()}  (id {new_id})\n")
for w, seq in splits.items():
    print(f"{w:6} x{word_freqs[w]:<3}  {render(seq, vocab):12}  ids={list(seq)}")

### Merges 2, 3, 4

Now just repeat: recount pairs on the **updated** splits, merge the winner, repeat.

Watch merge 3 especially — `h` merges with `ug`, a token that did not exist three steps ago. **Merges build on merges.** That dependency is why order matters later.

In [ ]:
for step in range(2, 5):
    pairs  = count_pairs(splits, word_freqs)
    best, n = pairs.most_common(1)[0]
    new_id  = 255 + step

    vocab[new_id] = vocab[best[0]] + vocab[best[1]]
    splits = {w: merge_seq(s, best, new_id) for w, s in splits.items()}

    print(f"merge {step}:  {vocab[best[0]].decode()} + {vocab[best[1]].decode()} "
          f"-> {vocab[new_id].decode()}  (id {new_id}, count {n})")
    for w, seq in splits.items():
        print(f"    {w:6} {render(seq, vocab)}")
    print()

### What you actually saved

Two things, and the second is derived from the first:

In [ ]:
merges = {
    (117, 103): 256,   # u + g  -> ug
    (117, 110): 257,   # u + n  -> un
    (104, 256): 258,   # h + ug -> hug
    (112, 257): 259,   # p + un -> pun
}

print("id   token   built from")
print("-" * 34)
for i in [98, 103, 104, 110, 112, 115, 117]:
    print(f"{i:<4} {vocab[i].decode():<7} raw byte")
for (a, b), i in merges.items():
    print(f"{i:<4} {vocab[i].decode():<7} {vocab[a].decode()} + {vocab[b].decode()}")

---
## 2. The real trainer

Everything above, in one function. Note it returns `merges` as an ordered dict — Python preserves insertion order, and that order **is** the algorithm.

In [ ]:
def train_bpe(word_freqs, vocab_size, verbose=False):
    """Train byte-level BPE. Returns (merges, vocab).

    word_freqs: {chunk: count} — pre-tokenized chunks and how often each appears.
                Counting over *unique* chunks weighted by frequency (rather than
                over every occurrence) is what makes this tractable on real data.
    """
    vocab  = {i: bytes([i]) for i in range(256)}
    splits = {w: tuple(w.encode("utf-8")) for w in word_freqs}
    merges = {}

    for new_id in range(256, vocab_size):
        pairs = count_pairs(splits, word_freqs)
        if not pairs:
            break
        best, count = pairs.most_common(1)[0]
        if count < 2:                      # nothing repeats any more
            break

        merges[best]  = new_id
        vocab[new_id] = vocab[best[0]] + vocab[best[1]]
        splits = {w: merge_seq(s, best, new_id) for w, s in splits.items()}

        if verbose:
            print(f"{new_id}: {vocab[best[0]]!r} + {vocab[best[1]]!r} "
                  f"-> {vocab[new_id]!r}   (count {count})")

    return merges, vocab


merges, vocab = train_bpe(word_freqs, vocab_size=260, verbose=True)

Same four merges we derived by hand. Good.

---
## 3. Encoding

Convert text to bytes, then walk the merge list **in learned order**, applying each merge everywhere it fits.

In [ ]:
def encode(text, merges):
    seq = tuple(text.encode("utf-8"))
    for pair, new_id in merges.items():      # insertion order == learned order
        seq = merge_seq(seq, pair, new_id)
    return list(seq)


def encode_traced(text, merges, vocab):
    seq = tuple(text.encode("utf-8"))
    print(f"start            {render(seq, vocab)}")
    for (a, b), new_id in merges.items():
        before = seq
        seq = merge_seq(seq, (a, b), new_id)
        tag = "applied" if seq != before else "   -   "
        rule = f"{vocab[a].decode()}+{vocab[b].decode()}->{vocab[new_id].decode()}"
        print(f"{rule:<16} {render(seq, vocab):<12} {tag}")
    print(f"\nids: {list(seq)}")
    return list(seq)


encode_traced("hugs", merges, vocab)

Four bytes became two tokens: `hug` + `s`.

### Why byte-level saves you

`m` never appeared anywhere in the training corpus. Watch what happens anyway:

In [ ]:
ids = encode("mug", merges)
print("mug ->", ids, "->", render(ids, vocab))
print()
print("m is byte", ord("m"), "- it exists because ALL 256 bytes exist.")
print("There is no unknown token. There never can be.")

This is the entire reason for working on bytes rather than characters. A character-level BPE would have to either crash here or emit `<UNK>` and lose information.

---
## 4. Order matters (the trap)

The loop is **over merges on the outside, positions on the inside**:

```python
for pair, new_id in merges.items():   # learned order
    seq = merge_seq(seq, pair, new_id)
```

A very natural alternative is *"scan left to right, take the longest match at each position."* That's what WordPiece does — and it gives **different answers**.

Here's a minimal case. Two merges, learned in this order:

In [ ]:
# Hand-constructed to isolate the effect.
#   a=97  b=98  c=99
demo_merges = {
    (98, 99): 256,   # merge 1:  b + c -> bc
    (97, 98): 257,   # merge 2:  a + b -> ab
}
demo_vocab = {i: bytes([i]) for i in range(256)}
demo_vocab[256] = b"bc"
demo_vocab[257] = b"ab"

for (a, b), i in demo_merges.items():
    print(f"merge -> id {i}: {demo_vocab[a].decode()} + {demo_vocab[b].decode()} = {demo_vocab[i].decode()}")

In [ ]:
def encode_longest_match(text, vocab):
    """The WRONG rule for BPE: greedy longest match, left to right."""
    data = text.encode("utf-8")
    by_bytes = {v: k for k, v in vocab.items()}
    ids, i = [], 0
    while i < len(data):
        for j in range(len(data), i, -1):
            if data[i:j] in by_bytes:
                ids.append(by_bytes[data[i:j]])
                i = j
                break
    return ids


text = "abc"
a = encode(text, demo_merges)
b = encode_longest_match(text, demo_vocab)

print(f"BPE  (replay in order) : {a}  ->  {render(a, demo_vocab)}")
print(f"greedy longest match   : {b}  ->  {render(b, demo_vocab)}")
print()
print("Same vocabulary. Same input. Different output.")

The merge order carries the frequency ranking learned during training. Replaying it in order is what reproduces the segmentation your **model** was trained on. Get this wrong and everything still runs, output still looks plausible, and your tokenization silently disagrees with your training distribution.

### Why a single pass is enough

You never have to loop back and re-check earlier merges, because:

> **A merge can only create opportunities for *later* merges, never earlier ones.**

Merge #3 produces token `258`. Merges #1 and #2 were learned before `258` existed, so neither of their rules can possibly mention it. Nothing you create at step *k* can trigger a rule from step *j < k*.

That's what makes one forward pass provably complete.

---
## 5. Decoding

The easy direction: look up each id, glue the bytes together, decode text **once at the end**.

In [ ]:
def decode(ids, vocab):
    return b"".join(vocab[i] for i in ids).decode("utf-8")

ids = encode("hugs", merges)
print(ids, "->", repr(decode(ids, vocab)))

### The one trap

Some characters take more than one byte:

```
A    -> [65]                       1 byte
é    -> [195, 169]                 2 bytes
中   -> [228, 184, 173]             3 bytes
🙂   -> [240, 159, 153, 130]        4 bytes
```

And those bytes are **meaningless on their own**. Byte `195` isn't a character — it's a flag saying *"a 2-byte character starts here."*

BPE has no concept of a character. It merges frequent byte pairs, so a single character's bytes can easily end up in separate tokens. Let's hit the failure deliberately:

In [ ]:
trap_vocab = {i: bytes([i]) for i in range(256)}
trap_vocab[256] = b"ca"
tokens = [256, 102, 195, 169]          # "ca" "f" then é's two bytes, unmerged

print("--- WRONG: decode each token on its own ---")
try:
    print("".join(trap_vocab[t].decode("utf-8") for t in tokens))
except UnicodeDecodeError as e:
    print("UnicodeDecodeError:", e)

print()
print("--- RIGHT: join all bytes, then decode once ---")
joined = b"".join(trap_vocab[t] for t in tokens)
print("joined :", joined)
print("decoded:", joined.decode("utf-8"))

The difference between the two versions is one character: `""` versus `b""`.

```python
"".join(vocab[t].decode("utf-8") for t in ids)   # WRONG — decodes per token
b"".join(vocab[t] for t in ids).decode("utf-8")  # RIGHT — decodes once
```

**Where this actually bites:** streaming generation. You produce one token at a time and want to print it immediately:

```
token 256 -> b"ca"      complete   -> print "ca"
token 102 -> b"f"       complete   -> print "f"
token 195 -> b"\xc3"    INCOMPLETE -> hold, print nothing
token 169 -> b"\xa9"    now decodes -> print "é"
```

So keep a byte buffer and only flush what decodes cleanly. The lazy alternative is `errors="replace"`, which shows a temporary `�` — fine while debugging, ugly in a demo.

**One-line version: a token is a bag of bytes, not a piece of text.**

In [ ]:
# Round-trip on genuinely nasty input. ASCII will always pass; this is the test that matters.
cases = ["hello", "café", "中文测试", "🙂🙃", "tabs\tand\nnewlines", "  leading spaces"]

merges_r, vocab_r = train_bpe(Counter(" ".join(cases).split()), vocab_size=300)
for s in cases:
    ok = decode(encode(s, merges_r), vocab_r) == s
    print(f"{'PASS' if ok else 'FAIL'}  {s!r}")

---
## 6. Pre-tokenization

Before any of this, real tokenizers chop text into chunks with a regex, and **merges are never allowed to cross chunk boundaries**.

Without it, BPE happily learns `"the cat"` as one token because that pair is frequent — burning vocabulary on phrase-specific tokens that generalise terribly.

In [ ]:
try:
    import regex as re_mod
    GPT2_PATTERN = r"""'s|'t|'re|'ve|'m|'ll|'d| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""
except ImportError:                       # `re` has no \p{...}; ASCII-only fallback
    import re as re_mod
    GPT2_PATTERN = r"""'s|'t|'re|'ve|'m|'ll|'d| ?[A-Za-z]+| ?[0-9]+| ?[^\sA-Za-z0-9]+|\s+(?!\S)|\s+"""
    print("note: `regex` not installed, using ASCII fallback (pip install regex)")

pat = re_mod.compile(GPT2_PATTERN)

def pretokenize(text):
    return pat.findall(text)

pretokenize("Hello world's 123 things! Isn't it nice?")

Two details worth noticing in that output:

1. **Leading spaces attach to the following word.** `" world"` is one chunk, not `" "` + `"world"`. This is why `"The"` and `" The"` are *different tokens* in GPT-2 — a detail that surprises people writing prompts.
2. **Contractions split predictably.** `"'s"` and `"n't"` are handled explicitly by the first alternatives in the pattern.

Now the comparison — what merges do you learn with and without it?

In [ ]:
sample = ("the cat sat on the mat . the cat ate the rat . "
          "the dog sat on the log . the dog ate the frog . ") * 40

with_pre    = Counter(pretokenize(sample))
without_pre = Counter([sample])            # one giant chunk, no boundaries

m_with, v_with = train_bpe(with_pre,    vocab_size=276)
m_without, v_without = train_bpe(without_pre, vocab_size=276)

print("WITH pre-tokenization — first 12 merges:")
print("   ", [v_with[i].decode() for i in list(m_with.values())[:12]])
print()
print("WITHOUT pre-tokenization — first 12 merges:")
print("   ", [v_without[i].decode() for i in list(m_without.values())[:12]])

Look at the second list: tokens spanning spaces, and whole phrases swallowed into single entries. On a real corpus that's how you waste half your vocabulary on tokens that never generalise.

---
## 7. Real text, and the number that matters

The metric for a tokenizer is **bytes per token** — how much text you fit into one model position. Higher is better: it means longer effective context and fewer tokens to generate for the same output.

In [ ]:
SAMPLE = """Once upon a time, in a small village near the mountains, there lived a
young girl named Mira. Every morning she would walk to the river to fetch water for
her family. The river was cold and clear, and the stones at the bottom shone like
tiny stars. One day, while filling her bucket, Mira noticed something glittering
between the rocks. She reached down and pulled out a small silver key. The key was
old and covered in strange markings that she had never seen before. Mira showed the
key to her grandmother, who studied it for a long time without speaking. Finally her
grandmother said that the key belonged to a door in the old mill, a door that had
been locked for a hundred years. That evening, Mira walked to the mill with the key
in her pocket and her heart beating fast. The door was small and wooden and covered
in ivy. She pushed the ivy aside, found the lock, and turned the key.""" * 20

print(f"{len(SAMPLE):,} characters, {len(SAMPLE.encode()):,} bytes")

**Optional:** swap in real TinyStories if you have `datasets` installed. Skip this cell otherwise — everything below works on `SAMPLE`.

In [ ]:
# Optional — needs `pip install datasets` and a network connection.
try:
    from datasets import load_dataset
    ds = load_dataset("roneneldan/TinyStories", split="train", streaming=True)
    SAMPLE = "\n\n".join(x["text"] for _, x in zip(range(2000), ds))
    print(f"loaded TinyStories: {len(SAMPLE):,} characters")
except Exception as e:
    print(f"skipping TinyStories ({type(e).__name__}) - continuing with the built-in sample")

In [ ]:
chunks = Counter(pretokenize(SAMPLE))
print(f"{len(chunks):,} unique chunks from {sum(chunks.values()):,} total")
print("\nThis ratio is why we count over unique chunks weighted by frequency.")
print(f"Doing it the naive way would be ~{sum(chunks.values()) / len(chunks):.0f}x more work per merge.")

In [ ]:
import time

results = []
for vs in [300, 512, 1024, 2048]:
    t0 = time.time()
    m, v = train_bpe(chunks, vocab_size=vs)
    elapsed = time.time() - t0

    ids = []
    for chunk in pretokenize(SAMPLE):
        ids.extend(encode(chunk, m))

    results.append((vs, len(ids), len(SAMPLE.encode()) / len(ids), elapsed))

print(f"{'vocab':>7} {'tokens':>10} {'bytes/token':>13} {'train (s)':>11}")
print("-" * 45)
print(f"{256:>7} {len(SAMPLE.encode()):>10,} {1.00:>13.2f} {0.0:>11.1f}   <- raw bytes, no merges")
for vs, n, ratio, t in results:
    print(f"{vs:>7} {n:>10,} {ratio:>13.2f} {t:>11.1f}")

Every doubling of the vocabulary buys less than the last — that diminishing return is exactly why production tokenizers land around 32k–128k rather than continuing upward.

**If you're on the built-in sample, the last rows are identical.** That's vocabulary *saturation*, not a bug: the sample is a short passage repeated, so it contains only ~100 distinct chunks. Once every chunk has collapsed to a single token there is no pair left that occurs twice, and `train_bpe` stops early. Load real TinyStories in the cell above and the curve keeps climbing — which is itself the lesson that a tokenizer can only be as good as the diversity of what it was trained on.

### How does yours compare to GPT-2?

In [ ]:
# Optional — needs `pip install tiktoken`.
try:
    import tiktoken
    enc = tiktoken.get_encoding("gpt2")
    n = len(enc.encode(SAMPLE))
    print(f"GPT-2 BPE (vocab 50,257): {n:,} tokens, {len(SAMPLE.encode()) / n:.2f} bytes/token")
    print("\nYours is trained on a few hundred KB of one text; GPT-2 saw 40GB of web.")
    print("Closing most of that gap with 2k merges on in-domain text is the real lesson:")
    print("tokenizers are extremely sensitive to how well the training data matches the target.")
except ImportError:
    print("skipping - pip install tiktoken to compare against GPT-2")

---
## 8. Where this is slow, and what to do about it

Our trainer recounts **every** pair from scratch on **every** merge. That's `O(vocab_size x corpus)` and it's why the timings above grow the way they do.

Two fixes, in order of importance:

1. **Count over unique chunks weighted by frequency** — already done above, and by far the bigger win. On repetitive text it's the difference between seconds and hours.
2. **Update incrementally.** A merge only changes pair counts inside chunks that actually contained that pair. Track which chunks contain which pairs and touch only those.

Do (1) from the start. Reach for (2) only if it's still too slow — with (1) in place, on a few hundred MB, it usually isn't.

There's a matching optimisation on the encode side. Looping over all 50,000 merges per chunk is wasteful when almost none apply. Real implementations flip it: look at the pairs actually *present*, pick the one with the lowest merge rank, apply it, repeat until nothing is mergeable. Identical output, work proportional to merges that actually fire.

Keep the naive versions as the test oracle for the fast ones — same reference/fast split the mixers use.

---

## Exercises

1. **Ranked encoder.** Write the fast encoder described above and assert it matches `encode()` on 10,000 random strings.
2. **Incremental counting.** Make `train_bpe` update pair counts instead of recomputing them, and measure the speedup at vocab 4096.
3. **Special tokens.** Add `<|endoftext|>` such that it can never be produced by merging, and always encodes to exactly one id.
4. **Domain sensitivity.** Train on TinyStories and measure bytes/token on Python source. Then train on Python and measure on TinyStories. The asymmetry is the argument for domain-matched tokenizers.
5. **WordPiece.** Change the merge criterion from `count(a,b)` to `count(a,b) / (count(a) * count(b))` and compare the merges learned. That single line is most of the difference between GPT-2's tokenizer and BERT's.

---

## Next

- `demo/tokenizers/unigram.ipynb` — the opposite approach: start with a huge vocabulary and prune it down with EM. Used by T5 and Gemma.
- Move your implementation into `src/litterbox/data/tokenizer/bpe.py` behind the `Tokenizer` interface, so training can use it.